In [1]:
# ------------------------------------------------------------
# Sanity checks for BACI drought/precip indices ### NEW VERSION ###
#   A) Ref-period (1961–1990) per-month mean≈0, std≈1 (drought)
#   B) Precip–Drought correlation sign (expect negative)
# Paths and names match your composites layout.
# ------------------------------------------------------------
import os
import pandas as pd
import xarray as xr
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────
DIR = "ACI-Python/data/composites"
DROUGHT_PATH = os.path.join(DIR, "drought_index.nc")
PRECIP_PATH  = os.path.join(DIR, "precipitation_index.nc")

REF_START = "1961-01-31"   # month-end timestamps for alignment/printing
REF_END   = "1990-12-31"

SAVE_CSV = False  # set True to write CSVs next to the NetCDFs

# ── Helpers ───────────────────────────────────────────────────────────────
def _open_da(path, fallback_name=None):
    """Open as DataArray if possible; otherwise Dataset→DataArray."""
    try:
        da = xr.open_dataarray(path)
    except Exception:
        ds = xr.open_dataset(path)
        if fallback_name and fallback_name in ds:
            da = ds[fallback_name]
        else:
            # first variable
            vname = next(iter(ds.data_vars))
            da = ds[vname]
    # collapse 'step' if present
    if "step" in da.dims:
        da = da.mean(dim="step")
    return da

def _to_month_end_series(da: xr.DataArray) -> pd.Series:
    """Convert DataArray with time coord → pandas Series on month-end index."""
    s = da.to_series().dropna()
    # force month-end stamps for alignment with other series
    s.index = pd.to_datetime(s.index).to_period("M").to_timestamp("M")
    return s.sort_index()

# ── Load series ───────────────────────────────────────────────────────────
drought_da = _open_da(DROUGHT_PATH)                 # name may already be 'drought_index'
precip_da  = _open_da(PRECIP_PATH)                  # name may already be 'precipitation_index'

drought_s = _to_month_end_series(drought_da)
precip_s  = _to_month_end_series(precip_da)

# ── A) Reference-period per-month mean/std (calendar-month aware) ─────────
ref_slice = slice(pd.Timestamp(REF_START), pd.Timestamp(REF_END))

d_ref = drought_s.loc[ref_slice]
if d_ref.empty:
    raise RuntimeError("No drought data within the reference period for the sanity check.")

# Per calendar month (Jan..Dec)
ref_stats = (
    d_ref.groupby(d_ref.index.month)
         .agg(['mean', 'std'])
         .rename_axis('month')
         .rename(columns={'mean':'ref_mean', 'std':'ref_std'})
)

print("\n[Ref 1961–1990] Drought per-month mean/std (want mean≈0, std≈1):")
print(ref_stats.round(3).to_string())

# Deviation diagnostics
ref_stats['abs_mean'] = ref_stats['ref_mean'].abs()
ref_stats['std_dev']  = (ref_stats['ref_std'] - 1.0).abs()

mean_tol = 0.15   # acceptable |mean| within ref period
std_tol  = 0.20   # acceptable |std-1| within ref period

bad_mean_months = ref_stats.index[ref_stats['abs_mean'] > mean_tol].tolist()
bad_std_months  = ref_stats.index[ref_stats['std_dev']  > std_tol ].tolist()

if bad_mean_months:
    print(f"\n⚠️  Months with |mean| > {mean_tol}: {bad_mean_months}")
else:
    print(f"\n✅ All months have |mean| ≤ {mean_tol}")

if bad_std_months:
    print(f"⚠️  Months with |std−1| > {std_tol}: {bad_std_months}")
else:
    print(f"✅ All months have |std−1| ≤ {std_tol}")

# Optional CSV
if SAVE_CSV:
    out_csv = os.path.join(DIR, "drought_ref1961_1990_per_month_stats.csv")
    ref_stats.to_csv(out_csv, float_format="%.6f")
    print(f"💾 Saved: {out_csv}")

# ── B) Precip–Drought correlation (expect negative) ───────────────────────
p_aligned, d_aligned = precip_s.align(drought_s, join="inner")
if len(p_aligned) == 0:
    raise RuntimeError("No overlapping months between precipitation and drought series.")
corr_pd = p_aligned.corr(d_aligned)

print("\nPrecip–Drought correlation on overlap (expect negative): "
      f"{corr_pd:.3f} (n={len(p_aligned)})")

# Extra: print full-period summaries (optional but handy)
def _summ(s, label):
    print(f"\n{label} — full period summary:")
    print(f"  start: {s.index.min():%Y-%m}, end: {s.index.max():%Y-%m}, n={len(s)}")
    print(f"  mean: {s.mean():.3f}, std: {s.std():.3f}, min: {s.min():.3f}, max: {s.max():.3f}")

_summaries = True
if _summaries:
    _summ(drought_s, "Drought")
    _summ(precip_s,  "Precipitation")



[Ref 1961–1990] Drought per-month mean/std (want mean≈0, std≈1):
       ref_mean  ref_std
month                   
1           0.0    1.017
2           0.0    1.017
3          -0.0    1.017
4          -0.0    1.017
5          -0.0    1.017
6          -0.0    1.017
7           0.0    1.017
8          -0.0    1.017
9          -0.0    1.017
10          0.0    1.017
11         -0.0    1.017
12          0.0    1.017

✅ All months have |mean| ≤ 0.15
✅ All months have |std−1| ≤ 0.2

Precip–Drought correlation on overlap (expect negative): -0.387 (n=768)

Drought — full period summary:
  start: 1961-01, end: 2024-12, n=768
  mean: -0.006, std: 1.089, min: -2.599, max: 4.483

Precipitation — full period summary:
  start: 1961-01, end: 2024-12, n=768
  mean: 0.095, std: 1.102, min: -2.471, max: 6.599


In [2]:
# ------------------------------------------------------------
# Sanity checks for BACI drought/precip indices ### OLD VERSION ###
#   A) Ref-period (1961–1990) per-month mean≈0, std≈1 (drought)
#   B) Precip–Drought correlation sign (expect negative)
# Paths and names match your composites layout.
# ------------------------------------------------------------
import os
import pandas as pd
import xarray as xr
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────
DIR = "ACI-Python/data/composites"
DROUGHT_PATH = os.path.join(DIR, "drought_index_old.nc")
PRECIP_PATH  = os.path.join(DIR, "precipitation_index_old.nc")

REF_START = "1961-01-31"   # month-end timestamps for alignment/printing
REF_END   = "1990-12-31"

SAVE_CSV = False  # set True to write CSVs next to the NetCDFs

# ── Helpers ───────────────────────────────────────────────────────────────
def _open_da(path, fallback_name=None):
    """Open as DataArray if possible; otherwise Dataset→DataArray."""
    try:
        da = xr.open_dataarray(path)
    except Exception:
        ds = xr.open_dataset(path)
        if fallback_name and fallback_name in ds:
            da = ds[fallback_name]
        else:
            # first variable
            vname = next(iter(ds.data_vars))
            da = ds[vname]
    # collapse 'step' if present
    if "step" in da.dims:
        da = da.mean(dim="step")
    return da

def _to_month_end_series(da: xr.DataArray) -> pd.Series:
    """Convert DataArray with time coord → pandas Series on month-end index."""
    s = da.to_series().dropna()
    # force month-end stamps for alignment with other series
    s.index = pd.to_datetime(s.index).to_period("M").to_timestamp("M")
    return s.sort_index()

# ── Load series ───────────────────────────────────────────────────────────
drought_da = _open_da(DROUGHT_PATH)                 # name may already be 'drought_index'
precip_da  = _open_da(PRECIP_PATH)                  # name may already be 'precipitation_index'

drought_s = _to_month_end_series(drought_da)
precip_s  = _to_month_end_series(precip_da)

# ── A) Reference-period per-month mean/std (calendar-month aware) ─────────
ref_slice = slice(pd.Timestamp(REF_START), pd.Timestamp(REF_END))

d_ref = drought_s.loc[ref_slice]
if d_ref.empty:
    raise RuntimeError("No drought data within the reference period for the sanity check.")

# Per calendar month (Jan..Dec)
ref_stats = (
    d_ref.groupby(d_ref.index.month)
         .agg(['mean', 'std'])
         .rename_axis('month')
         .rename(columns={'mean':'ref_mean', 'std':'ref_std'})
)

print("\n[Ref 1961–1990] Drought per-month mean/std (want mean≈0, std≈1):")
print(ref_stats.round(3).to_string())

# Deviation diagnostics
ref_stats['abs_mean'] = ref_stats['ref_mean'].abs()
ref_stats['std_dev']  = (ref_stats['ref_std'] - 1.0).abs()

mean_tol = 0.15   # acceptable |mean| within ref period
std_tol  = 0.20   # acceptable |std-1| within ref period

bad_mean_months = ref_stats.index[ref_stats['abs_mean'] > mean_tol].tolist()
bad_std_months  = ref_stats.index[ref_stats['std_dev']  > std_tol ].tolist()

if bad_mean_months:
    print(f"\n⚠️  Months with |mean| > {mean_tol}: {bad_mean_months}")
else:
    print(f"\n✅ All months have |mean| ≤ {mean_tol}")

if bad_std_months:
    print(f"⚠️  Months with |std−1| > {std_tol}: {bad_std_months}")
else:
    print(f"✅ All months have |std−1| ≤ {std_tol}")

# Optional CSV
if SAVE_CSV:
    out_csv = os.path.join(DIR, "drought_ref1961_1990_per_month_stats.csv")
    ref_stats.to_csv(out_csv, float_format="%.6f")
    print(f"💾 Saved: {out_csv}")

# ── B) Precip–Drought correlation (expect negative) ───────────────────────
p_aligned, d_aligned = precip_s.align(drought_s, join="inner")
if len(p_aligned) == 0:
    raise RuntimeError("No overlapping months between precipitation and drought series.")
corr_pd = p_aligned.corr(d_aligned)

print("\nPrecip–Drought correlation on overlap (expect negative): "
      f"{corr_pd:.3f} (n={len(p_aligned)})")

# Extra: print full-period summaries (optional but handy)
def _summ(s, label):
    print(f"\n{label} — full period summary:")
    print(f"  start: {s.index.min():%Y-%m}, end: {s.index.max():%Y-%m}, n={len(s)}")
    print(f"  mean: {s.mean():.3f}, std: {s.std():.3f}, min: {s.min():.3f}, max: {s.max():.3f}")

_summaries = True
if _summaries:
    _summ(drought_s, "Drought")
    _summ(precip_s,  "Precipitation")



[Ref 1961–1990] Drought per-month mean/std (want mean≈0, std≈1):
       ref_mean  ref_std
month                   
1           0.0    0.467
2          -0.0    0.477
3          -0.0    0.488
4           0.0    0.497
5          -0.0    0.505
6           0.0    0.507
7          -0.0    0.505
8           0.0    0.497
9           0.0    0.487
10         -0.0    0.476
11          0.0    0.466
12          0.0    0.457

✅ All months have |mean| ≤ 0.15
⚠️  Months with |std−1| > 0.2: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Precip–Drought correlation on overlap (expect negative): 0.108 (n=768)

Drought — full period summary:
  start: 1961-01, end: 2024-12, n=768
  mean: 0.067, std: 0.560, min: -0.954, max: 1.851

Precipitation — full period summary:
  start: 1961-01, end: 2024-12, n=768
  mean: 0.073, std: 0.847, min: -1.872, max: 4.528


In [3]:
# baci_sanity_checks.py ### NEW VERSION ###
# ------------------------------------------------------------
# Diagnostic checks for BACI drought/precip indices
# Run *after* your composites are saved to disk.
# ------------------------------------------------------------
import os
import pandas as pd
import xarray as xr
import numpy as np

DIR = "ACI-Python/data/composites"
DROUGHT_PATH = os.path.join(DIR, "drought_index.nc")
PRECIP_PATH  = os.path.join(DIR, "precipitation_index.nc")

REF_START, REF_END = "1961-01-31", "1990-12-31"

# ── Helpers ──────────────────────────────────────────────────────────────
def _open_series(path, name=None):
    da = xr.open_dataarray(path)
    if "step" in da.dims:
        da = da.mean("step")
    other = [d for d in da.dims if d != "time"]
    if other:
        da = da.mean(dim=other, skipna=True)
    s = da.to_series().dropna()
    s.index = pd.to_datetime(s.index).to_period("M").to_timestamp("M")
    s.name = name
    return s

def _summ(s, label):
    print(f"\n{label} — summary:")
    print(f"  start: {s.index.min():%Y-%m}, end: {s.index.max():%Y-%m}, n={len(s)}")
    print(f"  mean: {s.mean():.3f}, std: {s.std():.3f}, min: {s.min():.3f}, max: {s.max():.3f}")

# ── Load data ────────────────────────────────────────────────────────────
drought_s = _open_series(DROUGHT_PATH, "Drought")
precip_s  = _open_series(PRECIP_PATH,  "Precip")

# ── 1. Ref-period calibration ───────────────────────────────────────────
ref_slice = slice(pd.Timestamp(REF_START), pd.Timestamp(REF_END))
d_ref = drought_s.loc[ref_slice]

ref_stats = (
    d_ref.groupby(d_ref.index.month)
         .agg(['mean','std'])
         .rename_axis('month')
         .rename(columns={'mean':'ref_mean','std':'ref_std'})
)

print("\n[Ref 1961–1990] Drought per-month mean/std:")
print(ref_stats.round(3).to_string())

# ── 2. Global correlation ────────────────────────────────────────────────
p_aligned, d_aligned = precip_s.align(drought_s, join="inner")
corr = p_aligned.corr(d_aligned)
print(f"\nPrecip–Drought correlation (expect negative): {corr:.3f} (n={len(p_aligned)})")

# ── 3. Threshold sensitivity (0.5 vs 1.0 mm/day) ─────────────────────────
# NB: requires drought_bis recomputation; here we just show how to reload
try:
    from aci.components.drought_bis import DroughtComponent
    d05 = DroughtComponent("ACI-Python/data/required_data/precipitation",
                           "ACI-Python/data/required_data/mask_BEL.nc")
    d05._DRY_THRESHOLD_M = 0.0005
    di05 = d05.calculate_component(("1961-01-01","1990-12-31"), area=True).compute()
    d05_s = di05.to_series()
    d05_s.index = pd.to_datetime(d05_s.index).to_period("M").to_timestamp("M")
    print(f"\nThreshold test: corr(0.5 mm vs 1.0 mm) = {d05_s.corr(drought_s):.3f}")
except Exception as e:
    print(f"\n⚠️ Threshold sensitivity skipped (needs drought_bis): {e}")

# ── 4. Seasonal correlation (JJA/DJF) ────────────────────────────────────
season = pd.Series(drought_s.index.month, index=drought_s.index).map({
    12:"DJF",1:"DJF",2:"DJF",3:"MAM",4:"MAM",5:"MAM",
    6:"JJA",7:"JJA",8:"JJA",9:"SON",10:"SON",11:"SON"
})

for seas in ["DJF","JJA"]:
    mask = season==seas
    c = precip_s[mask].corr(drought_s[mask])
    print(f"Precip–Drought corr in {seas}: {c:.3f} (n={mask.sum()})")

# ── 5. Summaries ─────────────────────────────────────────────────────────
_summ(drought_s, "Drought (full)")
_summ(precip_s,  "Precipitation (full)")



[Ref 1961–1990] Drought per-month mean/std:
       ref_mean  ref_std
month                   
1           0.0    1.017
2           0.0    1.017
3          -0.0    1.017
4          -0.0    1.017
5          -0.0    1.017
6          -0.0    1.017
7           0.0    1.017
8          -0.0    1.017
9          -0.0    1.017
10          0.0    1.017
11         -0.0    1.017
12          0.0    1.017

Precip–Drought correlation (expect negative): -0.387 (n=768)

Threshold test: corr(0.5 mm vs 1.0 mm) = 0.943
Precip–Drought corr in DJF: -0.450 (n=192)
Precip–Drought corr in JJA: -0.335 (n=192)

Drought (full) — summary:
  start: 1961-01, end: 2024-12, n=768
  mean: -0.006, std: 1.089, min: -2.599, max: 4.483

Precipitation (full) — summary:
  start: 1961-01, end: 2024-12, n=768
  mean: 0.095, std: 1.102, min: -2.471, max: 6.599


In [1]:
# baci_sanity_checks.py ### OLD VERSION ###
# ------------------------------------------------------------
# Diagnostic checks for BACI drought/precip indices
# Run *after* your composites are saved to disk.
# ------------------------------------------------------------
import os
import pandas as pd
import xarray as xr
import numpy as np

DIR = "ACI-Python/data/composites"
DROUGHT_PATH = os.path.join(DIR, "drought_index_old.nc")
PRECIP_PATH  = os.path.join(DIR, "precipitation_index_old.nc")

REF_START, REF_END = "1961-01-31", "1990-12-31"

# ── Helpers ──────────────────────────────────────────────────────────────
def _open_series(path, name=None):
    da = xr.open_dataarray(path)
    if "step" in da.dims:
        da = da.mean("step")
    other = [d for d in da.dims if d != "time"]
    if other:
        da = da.mean(dim=other, skipna=True)
    s = da.to_series().dropna()
    s.index = pd.to_datetime(s.index).to_period("M").to_timestamp("M")
    s.name = name
    return s

def _summ(s, label):
    print(f"\n{label} — summary:")
    print(f"  start: {s.index.min():%Y-%m}, end: {s.index.max():%Y-%m}, n={len(s)}")
    print(f"  mean: {s.mean():.3f}, std: {s.std():.3f}, min: {s.min():.3f}, max: {s.max():.3f}")

# ── Load data ────────────────────────────────────────────────────────────
drought_s = _open_series(DROUGHT_PATH, "Drought")
precip_s  = _open_series(PRECIP_PATH,  "Precip")

# ── 1. Ref-period calibration ───────────────────────────────────────────
ref_slice = slice(pd.Timestamp(REF_START), pd.Timestamp(REF_END))
d_ref = drought_s.loc[ref_slice]

ref_stats = (
    d_ref.groupby(d_ref.index.month)
         .agg(['mean','std'])
         .rename_axis('month')
         .rename(columns={'mean':'ref_mean','std':'ref_std'})
)

print("\n[Ref 1961–1990] Drought per-month mean/std:")
print(ref_stats.round(3).to_string())

# ── 2. Global correlation ────────────────────────────────────────────────
p_aligned, d_aligned = precip_s.align(drought_s, join="inner")
corr = p_aligned.corr(d_aligned)
print(f"\nPrecip–Drought correlation (expect negative): {corr:.3f} (n={len(p_aligned)})")

# ── 3. Threshold sensitivity (0.5 vs 1.0 mm/day) ─────────────────────────
# NB: requires drought_bis recomputation; here we just show how to reload
try:
    from aci.components.drought_bis import DroughtComponent
    d05 = DroughtComponent("ACI-Python/data/required_data/precipitation",
                           "ACI-Python/data/required_data/mask_BEL.nc")
    d05._DRY_THRESHOLD_M = 0.0005
    di05 = d05.calculate_component(("1961-01-01","1990-12-31"), area=True).compute()
    d05_s = di05.to_series()
    d05_s.index = pd.to_datetime(d05_s.index).to_period("M").to_timestamp("M")
    print(f"\nThreshold test: corr(0.5 mm vs 1.0 mm) = {d05_s.corr(drought_s):.3f}")
except Exception as e:
    print(f"\n⚠️ Threshold sensitivity skipped (needs drought_bis): {e}")

# ── 4. Seasonal correlation (JJA/DJF) ────────────────────────────────────
season = pd.Series(drought_s.index.month, index=drought_s.index).map({
    12:"DJF",1:"DJF",2:"DJF",3:"MAM",4:"MAM",5:"MAM",
    6:"JJA",7:"JJA",8:"JJA",9:"SON",10:"SON",11:"SON"
})

for seas in ["DJF","JJA"]:
    mask = season==seas
    c = precip_s[mask].corr(drought_s[mask])
    print(f"Precip–Drought corr in {seas}: {c:.3f} (n={mask.sum()})")

# ── 5. Summaries ─────────────────────────────────────────────────────────
_summ(drought_s, "Drought (full)")
_summ(precip_s,  "Precipitation (full)")



[Ref 1961–1990] Drought per-month mean/std:
       ref_mean  ref_std
month                   
1           0.0    0.467
2          -0.0    0.477
3          -0.0    0.488
4           0.0    0.497
5          -0.0    0.505
6           0.0    0.507
7          -0.0    0.505
8           0.0    0.497
9           0.0    0.487
10         -0.0    0.476
11          0.0    0.466
12          0.0    0.457

Precip–Drought correlation (expect negative): 0.108 (n=768)

Threshold test: corr(0.5 mm vs 1.0 mm) = -0.045
Precip–Drought corr in DJF: 0.153 (n=192)
Precip–Drought corr in JJA: 0.004 (n=192)

Drought (full) — summary:
  start: 1961-01, end: 2024-12, n=768
  mean: 0.067, std: 0.560, min: -0.954, max: 1.851

Precipitation (full) — summary:
  start: 1961-01, end: 2024-12, n=768
  mean: 0.073, std: 0.847, min: -1.872, max: 4.528


In [2]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib import colors

# PDF output path
pdf_path = "ACI-Python/reports/methodological_note_drought_precip_resume.pdf"

# Document setup
doc = SimpleDocTemplate(pdf_path, pagesize=A4,
                        rightMargin=50, leftMargin=50,
                        topMargin=50, bottomMargin=50)
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="Justify", alignment=4, leading=14))

story = []

# Title
story.append(Paragraph("<b>Methodological Note: Drought and Precipitation Indices</b>", styles["Title"]))
story.append(Spacer(1, 12))

# Intro
intro = """
This note explains the rationale for adopting the <b>new methodology</b> in the calculation 
of drought and precipitation indices for the Belgian Actuarial Climate Index (BACI). 
A set of <i>sanity checks</i> were performed to compare the new and old approaches. 
The results clearly demonstrate that the new methodology provides indices that are 
statistically well-calibrated and physically consistent.
"""
story.append(Paragraph(intro, styles["Justify"]))
story.append(Spacer(1, 12))

# Section 1: Old vs New
section1 = """
<b>1. Old methodology:</b><br/>
The previous approach estimated drought using <i>yearly maxima of consecutive dry days</i> 
followed by linear interpolation to monthly values. Precipitation extremes were standardized 
using similar annual-to-monthly transformations. 
This method led to:<br/>
• Under-dispersion (monthly std dev ≈ 0.45–0.50 in the reference period).<br/>
• Spurious positive correlation with precipitation (+0.108), contrary to physical expectations.<br/>
• An underestimation of drought variability across months and years.
"""
story.append(Paragraph(section1, styles["Justify"]))
story.append(Spacer(1, 12))

section2 = """
<b>2. New methodology:</b><br/>
The revised approach computes drought indices directly from <i>monthly maxima of daily 
consecutive dry spells</i>, and precipitation indices from <i>rolling k-day totals</i>. 
Both indices are standardized by calendar month (or season) relative to the 1961–1990 
reference period. This ensures:<br/>
• Proper calibration: means ≈ 0 and std dev ≈ 1 across all months.<br/>
• Physically consistent results: strong negative correlation with precipitation (–0.387).<br/>
• A robust representation of extremes, avoiding artificial smoothing.
"""
story.append(Paragraph(section2, styles["Justify"]))
story.append(Spacer(1, 12))

# Section 3: Results table
story.append(Paragraph("<b>3. Sanity Check Results</b>", styles["Heading2"]))

data = [
    ["", "Old Method", "New Method"],
    ["Ref-period mean (per month)", "≈ 0", "≈ 0"],
    ["Ref-period std dev (per month)", "≈ 0.45–0.50", "≈ 1.02"],
    ["Precip–Drought correlation", "+0.108", "–0.387"],
    ["Interpretation", "Under-dispersed, inconsistent", "Well-calibrated, physically consistent"]
]

table = Table(data, colWidths=[150, 150, 150])
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
    ("ALIGN", (0, 0), (-1, -1), "CENTER"),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
]))
story.append(table)
story.append(Spacer(1, 20))

# Conclusion
conclusion = """
<b>Conclusion:</b><br/>
The new methodology is adopted for BACI because it produces indices that are both 
statistically robust (well-standardized in the reference period) and physically interpretable 
(negative drought–precipitation correlation). These improvements justify its use in the 
subsequent analysis and presentation of results.
"""
story.append(Paragraph(conclusion, styles["Justify"]))

# Build PDF
doc.build(story)

pdf_path


'ACI-Python/reports/methodological_note_drought_precip_resume.pdf'

In [3]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, LongTable, Table, TableStyle,
    PageBreak
)
from reportlab.lib import colors
from reportlab.pdfgen.canvas import Canvas

# ── PDF path ──────────────────────────────────────────────
pdf_path = "ACI-Python/reports/methodological_note_drought_precip.pdf"

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=A4,
    rightMargin=50, leftMargin=50,
    topMargin=60, bottomMargin=50
)

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="Justify", alignment=4, leading=14))
styles.add(ParagraphStyle(name="Small", parent=styles["Normal"], fontSize=8, leading=10))
styles.add(ParagraphStyle(name="H2", parent=styles["Heading2"], spaceBefore=6, spaceAfter=6))
styles.add(ParagraphStyle(name="H3", parent=styles["Heading3"], spaceBefore=4, spaceAfter=4))

def footer(canvas: Canvas, doc_):
    canvas.saveState()
    w, h = A4
    canvas.setFont("Helvetica", 9)
    canvas.setFillColor(colors.grey)
    canvas.drawRightString(w - 50, 20, f"Page {doc_.page}")
    canvas.restoreState()

def make_table(data, col_widths=None, header_bg=colors.lightgrey, font_size=8.5,
               striped=True, align="CENTER", grid_color=colors.grey, repeat_header=True):
    t = LongTable(data, colWidths=col_widths, repeatRows=1 if repeat_header else 0)
    style_cmds = [
        ("FONTSIZE", (0, 0), (-1, -1), font_size),
        ("BACKGROUND", (0, 0), (-1, 0), header_bg),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("ALIGN", (0, 0), (-1, -1), align),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("GRID", (0, 0), (-1, -1), 0.5, grid_color),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 3),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
    ]
    if striped and len(data) > 2:
        style_cmds += [("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.whitesmoke, colors.white])]
    t.setStyle(TableStyle(style_cmds))
    return t

story = []

# ── Title & Intro ──────────────────────────────────────────────
story.append(Paragraph("<b>Methodological Note: Drought and Precipitation Indices</b>", styles["Title"]))
story.append(Spacer(1, 10))

intro = """
This note explains the rationale for adopting the <b>new methodology</b> in the calculation 
of drought and precipitation indices for the Belgian Actuarial Climate Index (BACI). 
A set of <i>sanity checks</i> were performed to compare the new and old approaches. 
The results demonstrate that the new methodology provides indices that are statistically 
well-calibrated, robust across months and seasons, and consistent with physical expectations.
"""
story.append(Paragraph(intro, styles["Justify"]))
story.append(Spacer(1, 8))

# ── Section 1: Old ──────────────────────────────────────────────
section1 = """
<b>1. Old methodology:</b><br/>
The previous BACI prototype adopted an indirect procedure: <i>yearly maxima of consecutive dry days</i> 
were first computed, and then interpolated to provide pseudo-monthly values. Precipitation extremes 
were handled through a similar annual-to-monthly transformation. While computationally simple, this 
method had several important drawbacks:<br/>
• It produced <b>under-dispersed indices</b>, with reference-period monthly standard deviations of only ≈ 0.45–0.50.<br/>
• It generated a <b>spurious positive correlation</b> between drought and precipitation indices (+0.108), which contradicts 
the physical relationship where drought should be negatively associated with precipitation.<br/>
• It underestimated both <b>intra-annual variability</b> (differences between months) and <b>interannual variability</b>, 
thereby smoothing out extreme signals.
"""
story.append(Paragraph(section1, styles["Justify"]))
story.append(Spacer(1, 6))

# ── Section 2: New ──────────────────────────────────────────────
section2 = """
<b>2. New methodology:</b><br/>
The revised approach adopts a direct computation at the <b>monthly scale</b>:<br/>
• Drought indices are based on <i>monthly maxima of daily consecutive dry spells</i> (CDD).<br/>
• Precipitation indices are based on <i>rolling k-day totals</i> (short-term accumulation extremes).<br/><br/>
Both indices are standardized by calendar month (or season) relative to the 1961–1990 reference period. 
This ensures a <b>mean close to 0 and variance close to 1</b> across all months, preserving comparability. 
It also yields <b>physically consistent results</b>, with strong negative correlation between drought and precipitation 
(–0.387 overall), and a more faithful representation of extremes without artificial smoothing.
"""
story.append(Paragraph(section2, styles["Justify"]))
story.append(Spacer(1, 10))

# ── Section 3 Overview Table ────────────────────────────────────
story.append(Paragraph("<b>3. Sanity Check Results (Overview)</b>", styles["H2"]))
overview_data = [
    ["", "Old Method", "New Method"],
    ["Ref-period mean (per month)", "≈ 0", "≈ 0"],
    ["Ref-period std dev (per month)", "≈ 0.45–0.50", "≈ 1.02"],
    ["Precip–Drought correlation (all months)", "+0.108 (n=768)", "–0.387 (n=768)"],
    ["Interpretation", "Under-dispersed, inconsistent", "Well-calibrated, physically consistent"]
]
story.append(make_table(overview_data, col_widths=[180,170,170]))
story.append(PageBreak())

# ── Section 4 (NEW diagnostics) ─────────────────────────────────
story.append(Paragraph("<b>4. Detailed Diagnostics — NEW methodology</b>", styles["H2"]))
story.append(Paragraph("<b>4a) [Ref 1961–1990] Drought per-month mean/std</b>", styles["H3"]))
story.append(make_table([
    ["Month","ref_mean","ref_std"],
    ["1","0.0","1.017"], ["2","0.0","1.017"], ["3","-0.0","1.017"],
    ["4","-0.0","1.017"], ["5","-0.0","1.017"], ["6","-0.0","1.017"],
    ["7","0.0","1.017"], ["8","-0.0","1.017"], ["9","-0.0","1.017"],
    ["10","0.0","1.017"], ["11","-0.0","1.017"], ["12","0.0","1.017"],
], [70,120,120], header_bg=colors.whitesmoke))
story.append(Spacer(1,6))

story.append(Paragraph("<b>4b) Correlations</b>", styles["H3"]))
story.append(make_table([
    ["Metric","Value"],
    ["Precip–Drought correlation (all months)","−0.387 (n=768)"],
    ["Threshold test: corr(0.5 mm vs 1.0 mm)","0.943"],
    ["Precip–Drought corr in DJF","−0.450 (n=192)"],
    ["Precip–Drought corr in JJA","−0.335 (n=192)"],
],[260,260], header_bg=colors.whitesmoke))
story.append(Spacer(1,6))

story.append(Paragraph("<b>4c) Full-series summaries</b>", styles["H3"]))
story.append(make_table([
    ["Series","start","end","n","mean","std","min","max"],
    ["Drought","1961-01","2024-12","768","−0.006","1.089","−2.599","4.483"],
    ["Precipitation","1961-01","2024-12","768","0.095","1.102","−2.471","6.599"],
],[95,60,60,35,55,55,55,55], header_bg=colors.whitesmoke))
story.append(PageBreak())

# ── Section 5 (OLD diagnostics) ────────────────────────────────
story.append(Paragraph("<b>5. Detailed Diagnostics — OLD methodology</b>", styles["H2"]))
story.append(Paragraph("<b>5a) [Ref 1961–1990] Drought per-month mean/std</b>", styles["H3"]))
story.append(make_table([
    ["Month","ref_mean","ref_std"],
    ["1","0.0","0.467"], ["2","-0.0","0.477"], ["3","-0.0","0.488"],
    ["4","0.0","0.497"], ["5","-0.0","0.505"], ["6","0.0","0.507"],
    ["7","-0.0","0.505"], ["8","0.0","0.497"], ["9","0.0","0.487"],
    ["10","-0.0","0.476"], ["11","0.0","0.466"], ["12","0.0","0.457"],
],[70,120,120], header_bg=colors.whitesmoke))
story.append(Spacer(1,6))

story.append(Paragraph("<b>5b) Correlations</b>", styles["H3"]))
story.append(make_table([
    ["Metric","Value"],
    ["Precip–Drought correlation (all months)","+0.108 (n=768)"],
    ["Threshold test: corr(0.5 mm vs 1.0 mm)","−0.045"],
    ["Precip–Drought corr in DJF","0.153 (n=192)"],
    ["Precip–Drought corr in JJA","0.004 (n=192)"],
],[260,260], header_bg=colors.whitesmoke))
story.append(Spacer(1,6))

story.append(Paragraph("<b>5c) Full-series summaries</b>", styles["H3"]))
story.append(make_table([
    ["Series","start","end","n","mean","std","min","max"],
    ["Drought","1961-01","2024-12","768","0.067","0.560","−0.954","1.851"],
    ["Precipitation","1961-01","2024-12","768","0.073","0.847","−1.872","4.528"],
],[95,60,60,35,55,55,55,55], header_bg=colors.whitesmoke))
story.append(PageBreak())

# ── Section 6 & 7 ──────────────────────────────────────────────
story.append(Paragraph("<b>6. Interpretation</b>", styles["H2"]))
interp = """
The comparative evidence is clear. The <b>NEW methodology</b> succeeds in restoring statistical calibration: 
each month in the reference period has a standard deviation close to 1, which was lacking in the old design. 
Beyond calibration, it also <b>respects physical logic</b>, with a consistently negative correlation between 
precipitation and drought indicators. Seasonal values reinforce this point: in winter (DJF) the correlation 
is −0.450, and in summer (JJA) it is −0.335, both confirming that dry spells are linked with lower rainfall. 
Furthermore, robustness tests (threshold stability at 0.5 mm vs 1.0 mm) show very high agreement (corr = 0.943).<br/><br/>
By contrast, the <b>OLD methodology</b> suffers from structural flaws: it compresses variance, smooths extremes, 
and even produces <b>counter-intuitive positive correlations</b> between drought and precipitation, making its 
indices unreliable for actuarial or climate-risk applications.
"""
story.append(Paragraph(interp, styles["Justify"]))
story.append(Spacer(1, 10))

story.append(Paragraph("<b>7. One-page Recap</b>", styles["H2"]))
recap_data = [
    ["Metric","OLD","NEW"],
    ["Ref-period monthly std","≈ 0.45–0.50","≈ 1.02"],
    ["Precip–Drought corr (all)","+0.108","−0.387"],
    ["DJF corr","0.153","−0.450"],
    ["JJA corr","0.004","−0.335"],
    ["Threshold stability","−0.045","0.943"],
]
story.append(make_table(recap_data, [200,150,150], header_bg=colors.lightgrey))

# ── Build PDF ──────────────────────────────────────────────
doc.build(story, onFirstPage=footer, onLaterPages=footer)
print(pdf_path)


ACI-Python/reports/methodological_note_drought_precip.pdf


In [3]:
# build_methodological_note_all_indices_v2.py
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, KeepTogether
)
from reportlab.lib import colors
from reportlab.lib.units import mm

# Output
pdf_path = "ACI-Python/reports/methodological_note_all_indices.pdf"

# Document
doc = SimpleDocTemplate(
    pdf_path,
    pagesize=A4,
    rightMargin=50, leftMargin=50, topMargin=50, bottomMargin=50
)

# Styles
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="Justify", alignment=4, leading=14))
styles.add(ParagraphStyle(name="H3", fontSize=12, leading=14, spaceBefore=6, spaceAfter=4))
styles.add(ParagraphStyle(name="Cell", fontSize=9, leading=11))
styles.add(ParagraphStyle(name="Small", fontSize=9, leading=11))
Title = styles["Title"]
Body  = styles["Justify"]
H2    = styles["Heading2"]
H3    = styles["H3"]
Cell  = styles["Cell"]
Small = styles["Small"]

def p(text, style=Cell):
    return Paragraph(text, style)

def make_table(data, col_widths, header_row=0, body_font="Helvetica", header_font="Helvetica-Bold"):
    """Create a wrapped table with consistent styling."""
    tbl = Table(data, colWidths=col_widths, repeatRows=header_row+1, hAlign="CENTER")
    tbl.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,0), header_font),
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("TEXTCOLOR", (0,0), (-1,0), colors.black),
        ("FONTNAME", (0,1), (-1,-1), body_font),
        ("FONTSIZE", (0,0), (-1,-1), 9),
        ("ALIGN", (0,0), (-1,-1), "CENTER"),
        ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
        ("GRID", (0,0), (-1,-1), 0.4, colors.grey),
        ("WORDWRAP", (0,0), (-1,-1), True),
        ("LEFTPADDING", (0,0), (-1,-1), 4),
        ("RIGHTPADDING", (0,0), (-1,-1), 4),
        ("TOPPADDING", (0,0), (-1,-1), 4),
        ("BOTTOMPADDING", (0,0), (-1,-1), 4),
    ]))
    return tbl

story = []

# Title
story.append(Paragraph("<b>Methodological Note: BACI Indices — Rationale for the New Methodology</b>", Title))
story.append(Spacer(1, 8))

# Intro
intro = """
This methodological note documents the decision to adopt a <b>new, harmonized methodology</b> for all
BACI component indices: <i>Drought</i>, <i>Precipitation</i>, <i>T90</i> (warm), <i>T10</i> (cold), <i>Wind</i>,
and <i>Sea Level</i>. The new approach standardizes each index by calendar month (or season)
relative to the <b>1961–1990</b> reference period and prefers <i>direct monthly diagnostics</i> over
annual interpolation, ensuring well-calibrated and physically interpretable time series.
"""
story.append(Paragraph(intro, Body))
story.append(Spacer(1, 10))

# Global principles
principles = """
<b>Global principles applied to all indices</b><br/>
• <b>Monthly (or seasonal) standardization</b> with 1961–1990 means and standard deviations (ddof=0).<br/>
• <b>Area-mean before standardization</b> (mask-weighted when available) to avoid mixing scales.<br/>
• <b>Month-end timestamps</b> for clean alignment across components.<br/>
• <b>Direct diagnostics</b> (e.g., monthly CDD maxima, rolling k-day precipitation) rather than
annual values interpolated to months.<br/>
• <b>Sanity checks</b>: per-month mean≈0 and std≈1 in the reference period; physically expected
correlations/signs (e.g., Drought vs Precip negative); stability under reasonable parameter changes.
"""
story.append(Paragraph(principles, Body))
story.append(Spacer(1, 10))

# 1. Drought & Precipitation
story.append(Paragraph("<b>1. Drought & Precipitation</b>", H2))
drought_new = """
<b>Drought (new):</b> compute <i>monthly maxima of consecutive dry days (CDD)</i> directly from daily
precipitation (dry-day threshold 1.0&nbsp;mm/day), then standardize by calendar month using the 1961–1990 window.
This avoids artifacts from annual-to-monthly interpolation and preserves true monthly variability.
"""
precip_new = """
<b>Precipitation (new):</b> compute <i>monthly (or seasonal) maxima of rolling k-day totals</i> (default k=5)
and standardize by month/season over 1961–1990. This captures short-duration wet extremes consistently.
"""
story.append(Paragraph(drought_new, Body))
story.append(Spacer(1, 4))
story.append(Paragraph(precip_new, Body))
story.append(Spacer(1, 6))

# Drought & Precip sanity table (updated with NEW + OLD results you shared)
dp_data = [
    [p("<b>Check</b>", Small), p("<b>Old Method</b>", Small), p("<b>New Method (Observed)</b>", Small)],
    [p("Ref-period per-month mean", Small), p("≈ 0", Small), p("≈ 0 (all months)", Small)],
    [p("Ref-period per-month std", Small), p("≈ 0.46–0.51", Small), p("≈ 1.02 (all months ≈ 1.017)", Small)],
    [p("Precip–Drought correlation", Small), p("+0.108 (n=768)", Small), p("–0.387 (n=768)", Small)],
    [p("Seasonal correlations", Small), p("DJF: +0.153; JJA: +0.004", Small), p("DJF: –0.450; JJA: –0.335", Small)],
    [p("Threshold sensitivity (dry day)", Small), p("ρ(0.5mm vs 1.0mm) = –0.045", Small),
     p("ρ(0.5mm vs 1.0mm) = 0.943", Small)],
    [p("Drought (full) — summary", Small),
     p("mean 0.067, std 0.560, min –0.954, max 1.851", Small),
     p("mean –0.006, std 1.089, min –2.599, max 4.483", Small)],
    [p("Precip (full) — summary", Small),
     p("mean 0.073, std 0.847, min –1.872, max 4.528", Small),
     p("mean 0.095, std 1.102, min –2.471, max 6.599", Small)],
]
# A4 usable width ≈ 150 mm; give more space to the 3rd column
dp_widths = [42*mm, 48*mm, 60*mm]
dp_table = make_table(dp_data, dp_widths)
story.append(KeepTogether([Paragraph("<b>Sanity checks (Belgium, 1961–2024)</b>", H3), dp_table]))
story.append(Spacer(1, 8))

story.append(Paragraph(
    "These diagnostics show that the <b>new drought/precipitation methods</b> are both "
    "<i>statistically well-calibrated</i> (mean≈0, std≈1 by month) and <i>physically consistent</i> "
    "(negative drought–precipitation correlation).", Body))
story.append(Spacer(1, 10))

# Page break to avoid cramping section 2
story.append(PageBreak())

# 2. Temperature
story.append(Paragraph("<b>2. Temperature Extremes: T90 and T10</b>", H2))
temp_text = """
<b>Old:</b> daily extremes were derived with coarser temporal partitions and/or post-hoc interpolation, which could
smear within-month variability.<br/><br/>
<b>New:</b> compute <i>daytime and nighttime</i> extremes directly from hourly 2m temperature, using
rolling windows and <i>percentile thresholds</i> estimated within the reference period (1961–1990) with
day-of-year smoothing; then aggregate to monthly/seasonal exceedance <i>fractions</i> and standardize by calendar
month/season. This yields:<br/>
• Stable, comparable z-scores across months and years (mean≈0, std≈1 in reference).<br/>
• Less aliasing of heat/cold events around month boundaries.<br/>
• A clearer separation of warm (T90) and cold (T10) signals compatible with composite construction.
"""
story.append(Paragraph(temp_text, Body))
story.append(Spacer(1, 6))

temp_data = [
    [p("Check"), p("Old Method (Typical)"), p("New Method (Expected)")],
    [p("Ref-period per-month mean/std"), p("mean≈0, std<1 (under-dispersion)"), p("mean≈0, std≈1 (calibrated)")],
    [p("Day vs night handling"), p("Coarse partitions"), p("Explicit day/night, robust percentiles")],
    [p("Month boundary aliasing"), p("Possible"), p("Mitigated via daily/hourly handling")],
]
temp_widths = [48*mm, 48*mm, 48*mm]
temp_table = make_table(temp_data, temp_widths)
story.append(temp_table)
story.append(Spacer(1, 10))

# 3. Wind
story.append(Paragraph("<b>3. Wind</b>", H2))
wind_text = """
<b>Old:</b> wind exceedances were sometimes standardized after spatial aggregation or without consistent seasonal/monthly
z-scoring, leading to scale inconsistencies.<br/><br/>
<b>New:</b> compute daily <i>wind power</i> (½&nbsp;ρ&nbsp;U³ from 10m wind components), derive monthly (or seasonal)
<i>exceedance frequency</i> above reference-period, day-of-year-aware thresholds (e.g., mean+1.28σ), then
standardize by calendar month/season. This yields:<br/>
• Comparable z-scores across months (mean≈0, std≈1 in reference).<br/>
• Physically interpretable extremes of persistent windy conditions.<br/>
• Optional mask weighting and area-mean <i>before</i> standardization for consistency with other indices.
"""
story.append(Paragraph(wind_text, Body))
story.append(Spacer(1, 6))

wind_data = [
    [p("Check"), p("Old Method (Typical)"), p("New Method (Expected)")],
    [p("Ref-period per-month mean/std"), p("Inconsistent scaling"), p("mean≈0, std≈1")],
    [p("Aggregation order"), p("Standardize after area-mean"), p("Area-mean before standardization")],
    [p("Seasonality handling"), p("Partial"), p("Explicit monthly/seasonal z-scoring")],
]
wind_widths = [48*mm, 48*mm, 48*mm]
wind_table = make_table(wind_data, wind_widths)
story.append(wind_table)
story.append(Spacer(1, 10))

# 4. Sea Level
story.append(Paragraph("<b>4. Sea Level</b>", H2))
sl_text = """
<b>Old:</b> station series could be combined or standardized in ways that mixed spatial and temporal steps,
occasionally dulling variance or introducing seasonal biases.<br/><br/>
<b>New:</b> compute the <i>gauge-mean first</i> (across available PSMSL monthly RLR stations), then apply
<b>calendar-month z-scoring</b> over 1961–1990 (ddof=0). Timestamps are harmonized to month-end. This ensures:<br/>
• Spatial representativeness before scaling.<br/>
• Month-wise calibration (mean≈0, std≈1 in reference).<br/>
• Clean alignment with the other components for compositing and plotting.
"""
story.append(Paragraph(sl_text, Body))
story.append(Spacer(1, 10))

# 5. Summary of decisions — compact, wrapped table
story.append(Paragraph("<b>5. Summary of methodological decisions</b>", H2))
summary_data = [
    [p("<b>Index</b>"), p("<b>New Diagnostic</b>"), p("<b>Standardization</b>"), p("<b>Key Reason</b>")],
    [p("Drought"), p("Monthly max CDD from daily precip"), p("Per calendar month (1961–1990)"),
     p("Avoids annual interpolation; correct sign vs precip")],
    [p("Precipitation"), p("Monthly/seasonal max of k-day totals"), p("Per calendar month/season"),
     p("Captures short wet extremes consistently")],
    [p("T90 (warm)"), p("Day/night exceedance fractions"), p("Per calendar month/season"),
     p("Less aliasing; stable z-scores")],
    [p("T10 (cold)"), p("Day/night exceedance fractions"), p("Per calendar month/season"),
     p("Symmetry with T90; stable scaling")],
    [p("Wind"), p("Exceedance freq. of wind power (½ρU³)"), p("Per calendar month/season"),
     p("Physically interpretable persistence of wind")],
    [p("Sea Level"), p("Gauge-mean first; monthly z-scores"), p("Per calendar month (1961–1990)"),
     p("Spatial representativeness before scaling")],
]
# Fit within ~150 mm usable width; give more space to Diagnostic/Reason
summary_widths = [30*mm, 54*mm, 34*mm, 32*mm]
summary_table = make_table(summary_data, summary_widths)
story.append(summary_table)
story.append(Spacer(1, 8))

# Conclusion
conclusion = """
<b>Conclusion.</b> The <b>new methodology</b> is adopted for all BACI indices. It provides
month-wise (or seasonal) calibration against the 1961–1990 reference period, preserves true monthly variability,
and yields indices that are physically consistent and comparable across components. These improvements
strengthen the interpretability of the Belgian ACI and the robustness of downstream compositing.
"""
story.append(Paragraph(conclusion, Body))

# Build
doc.build(story)
print(pdf_path)


ACI-Python/reports/methodological_note_all_indices.pdf


In [2]:
# build_methodological_note_all_indices_auto.py
# Generates a multi-index methodological note PDF with live metrics from NetCDFs.

import os
import pandas as pd
import xarray as xr
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
)
from reportlab.lib import colors
from reportlab.lib.units import mm


# ---------- Config ----------
DIR = "ACI-Python/data/composites"
PATHS = {
    "Drought":       os.path.join(DIR, "drought_index.nc"),
    "Precipitation": os.path.join(DIR, "precipitation_index.nc"),
    "T90":           os.path.join(DIR, "t90_index.nc"),
    "T10":           os.path.join(DIR, "t10_index.nc"),
    "Wind":          os.path.join(DIR, "wind_index.nc"),
    "Sea Level":     os.path.join(DIR, "sealevel_index.nc"),
}
REF_START = "1961-01-31"
REF_END   = "1990-12-31"
PDF_PATH = "ACI-Python/reports/methodological_note_all_indices.pdf"


# ---------- Helpers ----------
def open_dataarray(path, fallback_name=None):
    if not os.path.exists(path):
        return None
    try:
        da = xr.open_dataarray(path)
    except Exception:
        ds = xr.open_dataset(path)
        if fallback_name and fallback_name in ds:
            da = ds[fallback_name]
        else:
            vname = next(iter(ds.data_vars))
            da = ds[vname]
    if "step" in da.dims:
        da = da.mean("step")
    return da

def to_month_end_series(da: xr.DataArray) -> pd.Series:
    if da is None:
        return None
    other = [d for d in da.dims if d != "time"]
    if other:
        da = da.mean(dim=other, skipna=True)
    s = da.to_series().dropna()
    s.index = pd.to_datetime(s.index).to_period("M").to_timestamp("M")
    return s.sort_index()

def per_month_ref_stats(s: pd.Series, ref_start: str, ref_end: str):
    if s is None or s.empty:
        return None
    ref = s.loc[slice(pd.Timestamp(ref_start), pd.Timestamp(ref_end))]
    if ref.empty:
        return None
    g = ref.groupby(ref.index.month).agg(['mean','std']).rename_axis('month')
    g = g.rename(columns={'mean':'ref_mean', 'std':'ref_std'})
    return g

def full_summary(s: pd.Series):
    if s is None or s.empty:
        return None
    return {
        "start": s.index.min().strftime("%Y-%m"),
        "end": s.index.max().strftime("%Y-%m"),
        "n": len(s),
        "mean": float(s.mean()),
        "std": float(s.std()),
        "min": float(s.min()),
        "max": float(s.max()),
    }

def seasonal_mask(index: pd.DatetimeIndex):
    return pd.Series(index.month, index=index).map({
        12:"DJF",1:"DJF",2:"DJF",
        3:"MAM",4:"MAM",5:"MAM",
        6:"JJA",7:"JJA",8:"JJA",
        9:"SON",10:"SON",11:"SON"
    })

def safe_corr(s1, s2, season: str | None = None):
    if s1 is None or s2 is None:
        return None, 0
    x, y = s1.align(s2, join="inner")
    if season is not None:
        seas = seasonal_mask(x.index)
        sel = seas == season
        x, y = x[sel], y[sel]
    if len(x) == 0:
        return None, 0
    return float(x.corr(y)), len(x)


# ---------- Load all series ----------
series = {}
ref_tables = {}
summaries = {}
for name, path in PATHS.items():
    da = open_dataarray(path)
    s = to_month_end_series(da) if da is not None else None
    series[name] = s
    ref_tables[name] = per_month_ref_stats(s, REF_START, REF_END) if s is not None else None
    summaries[name] = full_summary(s)

# ---------- Cross-checks ----------
pd_corr, pd_n       = safe_corr(series["Precipitation"], series["Drought"])
pd_corr_djf, n_djf  = safe_corr(series["Precipitation"], series["Drought"], "DJF")
pd_corr_jja, n_jja  = safe_corr(series["Precipitation"], series["Drought"], "JJA")
t_corr, t_n         = safe_corr(series["T90"], series["T10"])


# ---------- ReportLab setup ----------
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="Justify", alignment=4, leading=14))
styles.add(ParagraphStyle(name="H3", fontSize=12, leading=14, spaceBefore=6, spaceAfter=4))
styles.add(ParagraphStyle(name="Cell", fontSize=9, leading=11))
Title = styles["Title"]
Body  = styles["Justify"]
H2    = styles["Heading2"]
H3    = styles["H3"]
Cell  = styles["Cell"]

def p(txt, style=Cell):
    return Paragraph(txt, style)

def make_table(data, widths, header_row=0):
    tbl = Table(data, colWidths=widths, repeatRows=header_row+1, hAlign="CENTER")
    tbl.setStyle(TableStyle([
        ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
        ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
        ("TEXTCOLOR", (0,0), (-1,0), colors.black),
        ("FONTNAME", (0,1), (-1,-1), "Helvetica"),
        ("FONTSIZE", (0,0), (-1,-1), 9),
        ("ALIGN", (0,0), (-1,-1), "CENTER"),
        ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
        ("GRID", (0,0), (-1,-1), 0.4, colors.grey),
        ("WORDWRAP", (0,0), (-1,-1), True),
        ("PADDING", (0,0), (-1,-1), 4),
    ]))
    return tbl

def fmt_summary(s):
    if s is None:
        return "n/a"
    return f"{s['start']}→{s['end']}, n={s['n']}; mean {s['mean']:+.3f}; std {s['std']:.3f}; min {s['min']:+.3f}; max {s['max']:+.3f}"

def fmt_corr(val, n):
    if val is None:
        return "n/a"
    return f"{val:+.3f} (n={n})"


# ---------- Build story ----------
story = []
story.append(Paragraph("<b>BACI Indices — Methodological Note with Live Metrics</b>", Title))
story.append(Spacer(1, 8))

intro = """
This note documents the adoption of a <b>new, harmonized methodology</b> for all BACI indices:
<i>Drought</i>, <i>Precipitation</i>, <i>T90</i>, <i>T10</i>, <i>Wind</i>, and <i>Sea Level</i>.
Each index is standardized by calendar month (or season) relative to <b>1961–1990</b>, area-averaged
(mask-weighted if applicable) <b>before</b> standardization, and stored with month-end timestamps for alignment.
"""
story.append(Paragraph(intro, Body))
story.append(Spacer(1, 10))

# Calibration table
story.append(Paragraph("<b>1) Reference-period calibration (1961–1990)</b>", H2))
calib_rows = [[p("<b>Index</b>"), p("<b>Per-month mean≈0?</b>"), p("<b>Per-month std summary</b>")]]
for name in PATHS.keys():
    rt = ref_tables.get(name)
    if rt is None:
        mean_txt, std_txt = "n/a", "n/a"
    else:
        abs_mean_ok = (rt["ref_mean"].abs() <= 0.15).all()
        mean_txt = "Yes (all months)" if abs_mean_ok else "Check"
        std_dev = (rt["ref_std"] - 1.0).abs()
        std_txt = f"avg={rt['ref_std'].mean():.3f}; max|Δ|={std_dev.max():.3f}"
    calib_rows.append([p(name), p(mean_txt), p(std_txt)])
story.append(make_table(calib_rows, [36*mm, 50*mm, 64*mm]))
story.append(Spacer(1, 8))

# Cross-checks
story.append(Paragraph("<b>2) Cross-checks (physical consistency)</b>", H2))
cross_rows = [[p("<b>Pair</b>"), p("<b>Expectation</b>"), p("<b>Observed</b>")]]
cross_rows += [
    [p("Precip ↔ Drought"), p("Negative"), p(fmt_corr(pd_corr, pd_n))],
    [p("DJF: Precip ↔ Drought"), p("Negative"), p(fmt_corr(pd_corr_djf, n_djf))],
    [p("JJA: Precip ↔ Drought"), p("Negative"), p(fmt_corr(pd_corr_jja, n_jja))],
    [p("T90 ↔ T10"), p("Negative"), p(fmt_corr(t_corr, t_n))],
]
story.append(make_table(cross_rows, [45*mm, 45*mm, 60*mm]))
story.append(Spacer(1, 10))

# Full summaries
story.append(Paragraph("<b>3) Full-period summaries (1961–2024)</b>", H2))
sum_rows = [[p("<b>Index</b>"), p("<b>Summary</b>")]]
for name in PATHS.keys():
    sum_rows.append([p(name), p(fmt_summary(summaries.get(name)))])
story.append(make_table(sum_rows, [36*mm, 118*mm]))
story.append(Spacer(1, 10))

# Conclusion
conclusion = """
<b>Conclusion.</b> The new methodology yields indices that are <i>statistically well-calibrated</i>
(mean≈0, std≈1 by month in 1961–1990) and <i>physically consistent</i> (e.g., negative precip–drought correlations).
This consistency across Drought, Precipitation, T90, T10, Wind, and Sea Level supports a robust Belgian ACI composite.
"""
story.append(Paragraph(conclusion, Body))

# ---------- Build ----------
os.makedirs(os.path.dirname(PDF_PATH), exist_ok=True)
doc = SimpleDocTemplate(PDF_PATH, pagesize=A4, rightMargin=50, leftMargin=50, topMargin=50, bottomMargin=50)
doc.build(story)

print("✅ PDF saved at:", PDF_PATH)


✅ PDF saved at: ACI-Python/reports/methodological_note_all_indices.pdf
